# 02 · Backtest review
Walk-forward: each race is predicted using only races before it, then compared with simply predicting the grid order.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from f1pred import build, config, evaluate, plotting, probabilities
from f1pred.features import FEATURE_SETS, FEATURES, build_features

plotting.apply_theme()
entries, laps, conditions = build.load_processed()
feats = build_features(entries, laps, conditions)
latest_season = int(feats.loc[feats["FinishPosition"].notna(), "Season"].max())

In [ ]:
SEASON = latest_season
START_ROUND = 4

## Model vs grid order

In [ ]:
results = evaluate.backtest(feats, SEASON, START_ROUND)
display(evaluate.summarize(results))
display(results.round(3))

In [ ]:
x = list(range(len(results)))
labels = results["EventName"].str.replace(" Grand Prix", "")
fig, ax = plt.subplots(figsize=(11, 4.4))
series = [("GridSpearman", "Grid order", plotting.MUTED), ("Spearman", "Model", plotting.ACCENT)]
for col, name, color in series:
    ax.plot(x, results[col], color=color, marker="o", markeredgecolor=plotting.SURFACE, markeredgewidth=2, label=name)
if abs(results["Spearman"].iloc[-1] - results["GridSpearman"].iloc[-1]) > 0.05:
    for col, name, _ in series:
        ax.annotate(name, (x[-1], results[col].iloc[-1]), xytext=(10, 0), textcoords="offset points",
                    va="center", color=plotting.INK_SECONDARY)
ax.set_xticks(x, labels, rotation=35, ha="right")
ax.set(title=f"{SEASON}: how close each predicted order was (Spearman ρ, higher is better)", ylim=(0, 1))
ax.legend(loc="lower left")
plt.show()

## Feature sets

In [ ]:
compare = evaluate.compare(feats, SEASON, START_ROUND, list(FEATURE_SETS))
display(compare.round(3))

spearman = compare.loc["Spearman"].drop("GridBaseline").sort_values()
baseline = compare.loc["Spearman", "GridBaseline"]
fig, ax = plt.subplots(figsize=(8, 3.6))
best = spearman.idxmax()
colors = [plotting.ACCENT if name == best else plotting.DEEMPHASIS for name in spearman.index]
ax.barh(spearman.index, spearman.to_numpy(), height=0.55, color=colors)
for y, (name, value) in enumerate(spearman.items()):
    ax.annotate(f"{value:.3f}", (value, y), xytext=(-6, 0), textcoords="offset points", ha="right",
                va="center", fontsize=9, color="white" if name == best else plotting.INK)
ax.axvline(baseline, color=plotting.INK_SECONDARY, linewidth=1)
ax.set_ylim(-0.6, len(spearman) + 0.2)
ax.annotate(f"grid order {baseline:.3f}", (baseline, len(spearman) - 0.25), xytext=(6, 0),
            textcoords="offset points", ha="left", va="center", fontsize=9, color=plotting.INK_SECONDARY)
ax.set(title="Spearman ρ by feature set", xlim=(0, 1))
ax.grid(axis="y", visible=False)
plt.show()

## Probability calibration
If the model says 30% podium chance, those drivers should reach the podium about 30% of the time.

In [ ]:
prior = evaluate.season_race_indices(feats, SEASON - 1, START_ROUND)
races = list(evaluate.walk_forward(feats, prior + evaluate.season_race_indices(feats, SEASON, START_ROUND)))
scored = pd.concat(probabilities.rolling_probabilities(races, feats)[len(prior):])
scored["OnPodium"] = scored.groupby("RaceIdx")["FinishPosition"].rank(method="first") <= 3

bins = pd.cut(scored["PodiumPct"], [0, 5, 15, 30, 50, 70, 100], include_lowest=True)
reliability = scored.groupby(bins, observed=True).agg(
    Predicted=("PodiumPct", "mean"), Actual=("OnPodium", lambda s: s.mean() * 100), Drivers=("OnPodium", "size"))
display(reliability.round(1))

fig, ax = plt.subplots(figsize=(5.4, 5.2))
ax.plot([0, 100], [0, 100], color=plotting.AXIS, linewidth=1)
ax.annotate("perfectly calibrated", (62, 70), rotation=45, fontsize=9, color=plotting.MUTED)
ax.plot(reliability["Predicted"], reliability["Actual"], color=plotting.ACCENT, marker="o",
        markeredgecolor=plotting.SURFACE, markeredgewidth=2)
ax.set(title="Podium chance: predicted vs actual", xlabel="Predicted (%)", ylabel="Actual podium rate (%)",
       xlim=(0, 100), ylim=(0, 100))
ax.set_aspect("equal")
plt.show()